# 💳 Ejercicios: Stripe
# Transacciones de pago - importe, método y fraude

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales de Stripe.

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué método de pago
### nos deja más
### comisión total?

La comisión es el negocio de Stripe.
Volumen alto no siempre = más margen.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/stripe_pagos.csv'
)

ok = df[df['rechazada'] == 0]

r = (
  ok
  .groupby('metodo')
  .agg(
    comision=('comision_usd', 'sum'),
    volumen=('importe_usd', 'sum'),
    n=('transaccion_id', 'count'),
  )
  .round(0)
  .sort_values('comision',
    ascending=False)
)

print('Comisión por método:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué países tienen
### más tasa de
### fraude?

Tasa fraude = fraudes / transacciones.
Clave para ajustar el risk scoring.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/stripe_pagos.csv'
)

r = (
  df
  .groupby('pais')
  .agg(
    n=('transaccion_id', 'count'),
    fraude_pct=('es_fraude',
      lambda s: s.mean() * 100),
  )
  .round(2)
  .sort_values('fraude_pct',
    ascending=False)
)

print('Fraude por país:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿El importe alto
### multiplica el riesgo
### de fraude? (lift)

Troceamos importe en tramos y vemos
cómo sube la tasa de fraude.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/stripe_pagos.csv'
)

df['tramo'] = pd.qcut(
  df['importe_usd'], 4,
  labels=['Q1', 'Q2', 'Q3', 'Q4'],
)

base = df['es_fraude'].mean()

r = (
  df
  .groupby('tramo', observed=True)
  ['es_fraude']
  .mean()
)

lift = (r / base).round(2)

print('Lift de fraude por tramo:')
print(lift)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué sector mueve
### más volumen y qué
### ticket medio tiene?

Ticket medio = importe medio por
transacción. Define el tipo de cliente.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/stripe_pagos.csv'
)

r = (
  df
  .groupby('sector')
  .agg(
    volumen=('importe_usd', 'sum'),
    ticket=('importe_usd', 'mean'),
    n=('transaccion_id', 'count'),
  )
  .round(1)
  .sort_values('volumen',
    ascending=False)
)

print('Volumen y ticket por sector:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuánto dinero bueno
### perdemos por
### rechazar de más?

Transacciones rechazadas NO fraude =
fricción que nos cuesta ingresos.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/stripe_pagos.csv'
)

# rechazadas que NO eran fraude
falsas = df[
  (df['rechazada'] == 1)
  & (df['es_fraude'] == 0)
]

perdido = falsas['importe_usd'].sum()
com_perd = falsas['comision_usd'].sum()

print(f'Rechazos buenos: {len(falsas)}')
print(f'Volumen perdido: '
      f'{perdido:,.0f} USD')
print(f'Comisión perdida: '
      f'{com_perd:,.0f} USD')